In [50]:
%pip install openai -qqq

Note: you may need to restart the kernel to use updated packages.


In [51]:
import os
from pathlib import Path
from openai import OpenAI

In [17]:
api_key = os.getenv("MAKI_API_KEY")

In [18]:
client = OpenAI(
    base_url="https://maki.uni-mannheim.de/v1",
    api_key=api_key,
)

In [53]:
sorted([m.id for m in client.models.list()])

['fasttext-cc-de-embedding-300',
 'gemma4-12b',
 'gemma4-26b',
 'gemma4-26b-thinking',
 'jina-embeddings-v2-base-de',
 'llama-3.3-70b',
 'maki-chat',
 'minicpm5-2b',
 'minimax-m2.7',
 'ministral-3-14b',
 'parakeet-v3',
 'parakeet-v3-diarize',
 'qwen3-tts',
 'qwen3.5-122b',
 'qwen3.5-122b-thinking',
 'qwen3.6-36b',
 'qwen3.6-36b-thinking',
 'qwen3.8-27b',
 'qwen3.8-27b-short',
 'qwen3.8-27b-thinking',
 'voxtral-tts',
 'wildguard']

In [54]:
maki_model = "qwen3.5-122b-thinking"

In [33]:
# Filepaths
data_desc_fpath = Path("../data/dataset_description.txt")
dc_terms_fpath = Path("../templates/dc_terms.xml")

In [34]:
def load_fpaths(fpath: Path) -> str:
    with open(fpath, "r") as file:
        return file.read()

data_desc = load_fpaths(data_desc_fpath)
dc_terms_schema = load_fpaths(dc_terms_fpath)

In [38]:
print(data_desc)

README-DATEI REGIORADAR 2025
DOI: https://doi.org/10.7801/485



#ALLGEMEINE INFORMATIONEN

##Titel
RegioRadar 2025

##Autorin / Kontaktperson bei Fragen zum RegioRadar
Name: Astrid Kickum
Institution: Universität Mannheim - Zentrum für Lehrerbildung und Bildungsinnovation (ZLBI)
Email: astrid.kickum@uni-mannheim.de

##Autorin / Kontaktperson bei Fragen zum Projekt TransforMA
Name: Julia Derkau
ORCID: https://orcid.org/0000-0001-7758-054X
Institution: Universität Mannheim - Zentrum für Lehrerbildung und Bildungsinnovation (ZLBI)
Email: derkau@uni-mannheim.de

##Lizenz
Dieser Datensatz ist lizenziert unter der Creative Common Attribution 4.0 International (CC BY 4.0) Lizenz.
Die Daten dürfen frei verwendet, verändert und weitergegeben werden, solange die ursprünglichen Autorinnen korrekt genannt werden.



#BESCHREIBUNG DES DATENSATZES

Dieser Datensatz enthält die Ergebnisse einer Umfrage (n=754), die u.a. zum Ziel hatte, das aktuelle Wellbeing, Problemwahrnehmungen sowie Bedürfnisse d

In [39]:
print(dc_terms_schema)

<?xml version="1.0" encoding="UTF-8"?>
<xs:schema attributeFormDefault="unqualified" elementFormDefault="qualified" targetNamespace="http://purl.org/dc/terms/" xmlns="http://purl.org/dc/terms/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcmitype="http://purl.org/dc/dcmitype/" xmlns:xs="http://www.w3.org/2001/XMLSchema">
<xs:annotation>
<xs:documentation xml:lang="en">
DCterms XML Schema
XML Schema for http://purl.org/dc/terms/ namespace

Created 2008-02-11

Created by

Tim Cole (t-cole3@uiuc.edu)
Tom Habing (thabing@uiuc.edu)
Jane Hunter (jane@dstc.edu.au)
Pete Johnston (p.johnston@ukoln.ac.uk),
Carl Lagoze (lagoze@cs.cornell.edu)

This schema declares XML elements for the DC elements and
DC element refinements from the http://purl.org/dc/terms/ namespace.

It reuses the complexType dc:SimpleLiteral, imported from the dc.xsd
schema, which permits simple element content, and makes the xml:lang
attribute available.

This complexType permits the derivation of other complexTypes
whi

In [47]:
PROMPT = f"""Map the following <dataset_description> to Dublin Core terms by using the provided <dc_terms_schema>.
Return ONLY a valid XML and nothing else.

<dataset_description>
{data_desc}
</dataset_description>

<dc_terms_schema>
{dc_terms_schema}
</dc_terms_schema>"""

In [55]:
response = client.chat.completions.create(
    model=maki_model,
    messages=[
        {
            "role": "system",
            "content": "You are an expert in automating research data management tasks."
        },
        {
            "role": "user",
            "content": PROMPT
        }
    ],
)

In [49]:
print(response.choices[0].message.content)



<?xml version="1.0" encoding="UTF-8"?>
<metadata xmlns:dcterms="http://purl.org/dc/terms/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <dcterms:title xml:lang="de">RegioRadar 2025</dcterms:title>
  <dcterms:creator xml:lang="de">Kickum, Astrid</dcterms:creator>
  <dcterms:creator xml:lang="de">Derkau, Julia</dcterms:creator>
  <dcterms:contributor xml:lang="de">Universität Mannheim - Zentrum für Lehrerbildung und Bildungsinnovation (ZLBI)</dcterms:contributor>
  <dcterms:publisher xml:lang="de">Universität Mannheim</dcterms:publisher>
  <dcterms:created xsi:type="dcterms:W3CDTF">2025-05-01</dcterms:created>
  <dcterms:issued xsi:type="dcterms:W3CDTF">2025-09</dcterms:issued>
  <dcterms:description xml:lang="de">Dieser Datensatz enthält die Ergebnisse einer Umfrage (n=754), die u.a. zum Ziel hatte, das aktuelle Wellbeing, Problemwahrnehmungen sowie Bedürfnisse der Bürgerinnen und Bürger der Metropolregion Rhein-Neckar (MRN) zu e